# P10 — Classification des cultures agricoles par parcelle (RPG)

**Contexte métier.** Chambres d'agriculture, coopératives, assurance récolte, écologie (bandes fleuries, rotation) : connaître la culture d'une parcelle sans enquête terrain, détecter les changements de pratiques. Enjeu : le RPG déclaré sert de base aux aides PAC ; un modèle prédit la culture depuis la télédétection.

**Objectif de l'EDA.** Identifier la cible de classification (groupe de culture) et les régresseurs disponibles (surface, commune), quantifier le déséquilibre des classes.

**Données.** Registre parcellaire graphique (RPG) 2022 — Pays Basque, portail agriculture de la communauté Pays Basque (zabal-agriculture.opendata-paysbasque.fr). Licence ouverte. 72 284 parcelles sur 158 communes, 170 264 ha.

## 1. Téléchargement

Export CSV du RPG 2022 pour le Pays Basque :

In [ ]:
import urllib.request, os
os.makedirs("data", exist_ok=True)

URL = "https://zabal-agriculture.opendata-paysbasque.fr/api/explore/v2.1/catalog/datasets/rpg-2022-registre-parcellaire-graphique/exports/csv"
if not os.path.exists("data/rpg2022_paysbasque.csv"):
    urllib.request.urlretrieve(URL, "data/rpg2022_paysbasque.csv")
print(os.path.getsize("data/rpg2022_paysbasque.csv") / 1e6, "Mo")

## 2. Chargement

Séparateur `;`. Chaque ligne est une parcelle avec son code et son groupe de culture (cible) et sa surface.

In [ ]:
import pandas as pd, numpy as np

df = pd.read_csv("data/rpg2022_paysbasque.csv", sep=";", low_memory=False, encoding="utf-8-sig")
df.columns = [c.strip() for c in df.columns]
grp = "codification_cultures_principales_test_libelle_groupe_culture"
print(len(df), "parcelles,", df["insee"].nunique(), "communes")

## 3. Analyse

Distribution des groupes (déséquilibre des classes !), surface totale et médiane par groupe, et par commune.

In [ ]:
print("Distribution des groupes de culture :")
print(df[grp].value_counts().to_dict())
print()
print("Surface totale (ha) par groupe :")
print((df.groupby(grp)["surface_parcelle_m2"].sum() / 10000).round(0).sort_values(ascending=False).to_dict())
print()
print("Surface médiane (ha) par groupe :")
print((df.groupby(grp)["surface_parcelle_m2"].median() / 10000).round(3).to_dict())
print()
print("Top communes par surface :", (df.groupby("nom")["surface_parcelle_m2"].sum() / 10000).round(0).sort_values(ascending=False).head(5).to_dict())

## 4. Résultats observés (exécution du 16/10/2026)

- n parcelles : 72284
- n communes : 158
- surface totale ha : 170 264.00
- groupes dist : Prairies permanentes : 33756, Estives et landes : 20000, Maïs grain et ensilage : 5982, Divers : 4569, Prairies tempo…
- cultures top15 : Prairie permanente - herbe prédominante (ressources fourragères ligneuses absentes ou peu présentes) : 29299, Surface…
- surface ha par groupe : Estives et landes : 71 852.00, Prairies permanentes : 71 800.00, Maïs grain et ensilage : 13 669.00, Prairies tempora…
- surface mediane par groupe : Autres cultures industrielles : 0.54, Autres céréales : 1.50, Autres oléagineux : 2.00, Blé tendre : 1.71, Colza : 2.…
- n parcelles par commune top8 : Saint-Étienne-de-Baïgorry : 1928, Hasparren : 1919, Barcus : 1465, Irissarry : 1302, Came : 1300, Ossès : 1216, Mendi…
- surface ha par commune top8 : Larrau : 5 556.00, Estérençuby : 4 768.00, Banca : 4 635.00, Hasparren : 4 390.00, Saint-Étienne-de-Baïgorry : 4 155.…
- surf parc vs surface corr : 1.00
- part prairies pct : 0.00
- part cereales pct : 0.00

## 5. Variable cible et régresseurs

**Cible (classification multi-classe) : groupe de culture** (`code_group` / libellé groupe) d'une parcelle. Version difficile : culture précise (`code_cultu`), version simple : binaire prairie vs culture.

**Régresseurs :**
- `surface_parcelle_m2` : médiane 0,5 ha pour les prairies permanentes, surfaces distinctes pour le maïs
- `insee` / `nom` (commune) : la géographie structure les cultures (maïs dans la vallée de la Nive, estives en montagne)
- Pour un vrai modèle : la signature satellite (Sentinel-2, series temporelles NDVI) est le régresseur naturel — le RPG fournit la vérité terrain
- `surf_parc` (surface préablement déclarée)

**Pièges :**
- Classes très déséquilibrées : 33 756 parcelles de prairies permanentes contre 2 de légumineuses — regrouper en ~6-8 classes ou utiliser un score adapté (macro-F1)
- Le RPG ne contient pas de variables spectrales : sans imagerie satellite jointe, le modèle restera géographique (et moins intéressant) ; l'étape d'enrichissement fait partie du projet
- Le maïs domine les cultures à l'exception des prairies : bien séparer groupes (`Maïs grain et ensilage` = 5 982 parcelles)
- Une parcelle peut avoir plusieurs cultures en paquets — `code_cultu` principal retenu ici